 # 2.08a — Final LightGBM Training on All Data

 In 2.04 I trained the LightGBM models on 2019+2021 and held 2026 out so I could score
 them on data they had never seen. That gave me honest RMSE and MAE numbers. Now that
 the evaluation is done, I want the deployed models to actually learn from every row I
 have, including 2026. So here I refit all 6 LightGBM regressors on the full
 2019 + 2021 + 2026 dataset and save them as the production artifacts.

 No new performance numbers come out of this. The honest RMSE and MAE are in 2.04 and
 those do not change. I am just adding 2026 back into the training set now that it has
 finished serving as the holdout.

 Two things I am keeping exactly the same:
 - The hyperparameters are the exact values from the 2.02 Optuna search. Nothing gets
   re-tuned here.
 - n_estimators is fixed at 469 with no early stopping, since there is no validation
   set left to trigger it on a full-data fit.

In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path().resolve().parent))
from model_training.feature_prep import (
    TARGET_REGRESSION,
    TRAINING_YEARS,
    build_preprocessor,
    get_X_y,
    load_training_data,
)

 ## Setup

 Same horizons as 2.04. I name FULL_YEARS explicitly so it is obvious when reading the
 loop below that nothing is held out. The artifacts overwrite whatever 2.04 left in models/.

In [ ]:
HORIZONS = [60, 180, 360, 720, 1440, 2880]
HORIZON_LABELS = {60: "1hr", 180: "3hr", 360: "6hr",
                  720: "12hr", 1440: "24hr", 2880: "multi-day"}

FULL_YEARS  = TRAINING_YEARS   # [2019, 2021, 2026], no holdout withheld
USE_FLOAT32 = True

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Cores: {os.cpu_count()}  |  horizons: {[HORIZON_LABELS[h] for h in HORIZONS]}")
print(f"Training on: {FULL_YEARS}")
print(f"Writing artifacts -> {MODELS_DIR.resolve()}")

Cores: 32  |  horizons: ['1hr', '3hr', '6hr', '12hr', '24hr', 'multi-day']
Training on: [2019, 2021, 2026]
Writing artifacts -> C:\Users\clark\Desktop\citibike\models


 ## Hyperparameters

 These are copied verbatim from 2.02. I want one obvious place in this notebook to see
 what the production models actually use, so I hardcode them here rather than importing
 them from somewhere else.

In [ ]:
BEST_LGBM_PARAMS = dict(
    learning_rate     = 0.117,
    num_leaves        = 199,
    feature_fraction  = 0.85,
    bagging_fraction  = 0.733,
    min_child_samples = 54,
    bagging_freq      = 1,
    n_estimators      = 469,
    n_jobs            = -1,
    num_threads       = 32,
    random_state      = 42,
    verbose           = -1,
)

 ## Train all 6 horizons

 I load one horizon at a time to keep memory manageable. Each fit pulls about 14M rows,
 trains the pipeline, saves the artifact, then frees everything before moving to the next
 horizon. The saved artifact is a bare sklearn Pipeline so the serving layer just calls
 .predict(X) with no extra setup needed.

In [ ]:
for h in HORIZONS:
    label = HORIZON_LABELS[h]
    t0 = time.time()

    df = load_training_data(h, years=FULL_YEARS)
    X, y = get_X_y(df, TARGET_REGRESSION)
    if USE_FLOAT32:
        num_cols = X.select_dtypes(include=["float64", "float32"]).columns
        X[num_cols] = X[num_cols].astype("float32")
        y = y.astype("float32")
    span = f"{df['timestamp'].min().date()} -> {df['timestamp'].max().date()}"
    del df

    print(f"  {label:<10} {len(X):>11,} rows  ({span})  fitting ...", end=" ", flush=True)

    pipe = Pipeline([
        ("pre",   build_preprocessor("lightgbm")),
        ("model", LGBMRegressor(**BEST_LGBM_PARAMS)),
    ])
    pipe.fit(X, y)

    out = MODELS_DIR / f"lgbm_regression_{h}min.joblib"
    joblib.dump(pipe, out)
    print(f"saved -> {out.name}   ({time.time() - t0:.0f}s)")
    del X, y, pipe

  1hr         14,674,158 rows  (2019-01-01 -> 2026-06-18)  fitting ... saved -> lgbm_regression_60min.joblib   (537s)
  3hr         14,355,238 rows  (2019-01-01 -> 2026-06-17)  fitting ... saved -> lgbm_regression_180min.joblib   (499s)
  6hr         14,007,554 rows  (2019-01-01 -> 2026-06-17)  fitting ... saved -> lgbm_regression_360min.joblib   (494s)
  12hr        13,722,578 rows  (2019-01-01 -> 2026-06-17)  fitting ... saved -> lgbm_regression_720min.joblib   (518s)
  24hr        14,680,390 rows  (2019-01-01 -> 2026-06-17)  fitting ... saved -> lgbm_regression_1440min.joblib   (515s)
  multi-day   14,476,193 rows  (2019-01-01 -> 2026-06-16)  fitting ... saved -> lgbm_regression_2880min.joblib   (518s)


 ## Summary

 A quick check that all 6 files landed in models/ and look reasonable in size before
 moving on to 2.08b.

In [ ]:
print("\nLightGBM artifacts written:")
for h in HORIZONS:
    p = MODELS_DIR / f"lgbm_regression_{h}min.joblib"
    size_mb = p.stat().st_size / 1e6
    print(f"  {HORIZON_LABELS[h]:<10} {p.name}  ({size_mb:.1f} MB)")


LightGBM artifacts written:
  1hr        lgbm_regression_60min.joblib  (8.6 MB)
  3hr        lgbm_regression_180min.joblib  (8.7 MB)
  6hr        lgbm_regression_360min.joblib  (8.7 MB)
  12hr       lgbm_regression_720min.joblib  (8.7 MB)
  24hr       lgbm_regression_1440min.joblib  (8.7 MB)
  multi-day  lgbm_regression_2880min.joblib  (8.7 MB)


 ## Conclusion

 Six LightGBM regressors are now trained on the full 2019 + 2021 + 2026 dataset and
 saved to models/. The hyperparameters are frozen from the 2.02 Optuna search and
 nothing changed about how the model works. The RMSE and MAE numbers to quote are still
 the ones in 2.04. Run 2.08b next for the Linear regressors, then 2.08c for Logistic.